#  Single Slide Domain Detection w/ Cell Embeddings

This notebook will fine tune the scGPT-spatial model for the single slide domain detection task and generate output in a csv file.

Input: adata with spatial coordinates and ground truth domain labels

Output: adata with spatial coordinates and predicted domain labels / cell name, spatial coordinates and predicted domain labels

## Zero-shot Domain Detection

In [4]:
from typing import Iterable
import scanpy as sc
from sklearn.metrics import silhouette_score

def predict_domain_using_embedding(adata,
                                   dom_key: str = "domain",  # output label for clustering result
                                   method: str = "leiden", # "leiden" or "louvain"
                                   rep_key: str = "X_scGPT",  # adata.obsm[rep_key]: embedding
                                   target_clusters: int = 6,
                                   n_neighbors: int = 15,
                                   resolution_grid: Iterable[float] = (0.3, 0.5,
                                                                       0.8, 1.0,
                                                                       1.2),
                                   return_silhouette: bool = True,
                                   ):
    """
    predict domain using generated embedding.
    Returns:
      labels: np.ndarray[int] — Domain labels for each spot (encoded integers)
      best_res: Optional[float] — Resolution to use (if automatically selected)
      best_score: Optional[float] — Silhouette score (if calculated)
    Side Effects:
      - Writes adata.obsm[rep_key] = (n_spot, D)
      - Writes adata.obs[dom_key] = pandas.Categorical
    Depends:
      - adata.obsm[rep_key]
    """
    sc.pp.neighbors(adata, use_rep=rep_key, n_neighbors=n_neighbors)

    def _cluster_at(res):
        if method == "leiden":
            sc.tl.leiden(adata, resolution=res, key_added=f"{dom_key}_tmp")
        elif method == "louvain":
            sc.tl.louvain(adata, resolution=res, key_added=f"{dom_key}_tmp")
        else:
            raise ValueError("method must be 'leiden' or 'louvain'")
        return adata.obs[f"{dom_key}_tmp"].astype(
            "category").cat.codes.to_numpy()

    best_res, best_score, best_labels, best_clusters = None, -1.0, None, None
    if target_clusters is None:
        for res in resolution_grid:
            labels = _cluster_at(res)
            if len(np.unique(labels)) < 2:
                score = -1.0
            else:
                try:
                    score = silhouette_score(adata.obsm[rep_key], labels)
                except Exception:
                    score = -1.0
            if score > best_score:
                best_res, best_score, best_labels = res, score, labels
    else:
        for res in resolution_grid:
            labels = _cluster_at(res)
            try:
                score = silhouette_score(adata.obsm[rep_key], labels)
            except Exception:
                score = -1.0
            if best_clusters is None or best_clusters >= abs(
                    len(np.unique(labels)) - target_clusters):
                best_clusters = abs(len(np.unique(labels)) - target_clusters)
                best_res, best_score, best_labels = res, score, labels

    labels = best_labels if best_labels is not None else _cluster_at(
        resolution_grid[0])
    adata.obs[dom_key] = labels
    adata.obs[dom_key] = adata.obs[dom_key].astype("category")
    return labels, best_res, (best_score if return_silhouette else None)

## Adata

In [2]:
# load data
import numpy as np
import scanpy as sc

data_folder = '../../data/1_visium/'
adata = sc.read_h5ad(data_folder + '1_visium_scgpt_zero_shot.h5ad')
adata = adata[np.logical_not(adata.obs['ground_truth'].isna())]  #remove NAN
print(adata)
print(adata.obs.ground_truth.unique())

View of AnnData object with n_obs × n_vars = 4221 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'Region', 'ground_truth'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'X_scGPT', 'spatial'
['Layer1', 'Layer3', 'WM', 'Layer6', 'Layer5', 'Layer2', 'Layer4']
Categories (7, object): ['Layer1', 'Layer2', 'Layer3', 'Layer4', 'Layer5', 'Layer6', 'WM']


In [7]:
# run leiden for domain detection w. scGPT-spatial embeddings
labels, _, _ = predict_domain_using_embedding(adata, dom_key="domain_scgpt",
                                              method="leiden",
                                              rep_key="X_scGPT",
                                              target_clusters=6,
                                              n_neighbors=7)

# save domain detection results to adata / csv.
adata.write(data_folder + "1_visium_scgpt_zero_shot_domain_detection.h5ad")

print(adata)
print(adata.obs.domain_scgpt)

AnnData object with n_obs × n_vars = 4221 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'Region', 'ground_truth', 'domain_scgpt_tmp', 'domain_scgpt'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial', 'neighbors', 'leiden'
    obsm: 'X_scGPT', 'spatial'
    obsp: 'distances', 'connectivities'
AAACAACGAATAGTTC-1    0
AAACAAGTATCTCCCA-1    1
AAACAATCTACTAGCA-1    0
AAACACCAATAACTGC-1    6
AAACAGCTTTCAGAAG-1    3
                     ..
TTGTTGTGTGTCAAGA-1    5
TTGTTTCACATCCAGG-1    4
TTGTTTCATTAGTCTA-1    6
TTGTTTCCATACAACT-1    4
TTGTTTGTGTAAATTC-1    0
Name: domain_scgpt, Length: 4221, dtype: category
Categories (7, int8): [0, 1, 2, 3, 4, 5, 6]


## Npy

In [2]:
# load data

import numpy as np
import scanpy as sc

data_folder = '../../exploration/data/1_visium/'
adata = sc.read_h5ad(data_folder + '1_visium.h5ad')
adata.obsm['X_stofm'] = np.load(data_folder + "1_visium_stofm.npy")
print(adata)

AnnData object with n_obs × n_vars = 4221 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'Region', 'ground_truth', 'protocol'
    var: 'gene_ids', 'feature_types', 'genome', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial', 'X_stofm'


In [5]:
# run leiden for domain detection w. scGPT-spatial embeddings
labels, _, _ = predict_domain_using_embedding(adata, dom_key="domain_scgpt",
                                              method="leiden",
                                              rep_key="X_stofm",
                                              target_clusters=6,
                                              n_neighbors=7)

# save domain detection results to adata / csv.
adata.write(data_folder + "1_visium_stofm_zero_shot_domain_detection.h5ad")

print(adata)
print(adata.obs.domain_scgpt)

AnnData object with n_obs × n_vars = 4221 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'Region', 'ground_truth', 'protocol', 'domain_scgpt_tmp', 'domain_scgpt'
    var: 'gene_ids', 'feature_types', 'genome', 'gene_name'
    uns: 'spatial', 'neighbors', 'leiden'
    obsm: 'spatial', 'X_stofm'
    obsp: 'distances', 'connectivities'
AAACAACGAATAGTTC-1    2
AAACAAGTATCTCCCA-1    1
AAACAATCTACTAGCA-1    2
AAACACCAATAACTGC-1    4
AAACAGCTTTCAGAAG-1    0
                     ..
TTGTTGTGTGTCAAGA-1    0
TTGTTTCACATCCAGG-1    1
TTGTTTCATTAGTCTA-1    4
TTGTTTCCATACAACT-1    1
TTGTTTGTGTAAATTC-1    2
Name: domain_scgpt, Length: 4221, dtype: category
Categories (5, int8): [0, 1, 2, 3, 4]
